# 03 - Evaluation

Runs the 30-question golden set through both models. Retrieval metrics (Context Precision/Recall, MRR) come from the gold pages; generation metrics (Faithfulness, Relevancy, Correctness) come from an LLM judge (gemma3:4b, a different model from both generators).

In [1]:
%pip install -q ollama chromadb pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

In [3]:
from src.evaluation import run_evaluation

smol_rows = run_evaluation("smollm")
llama_rows = run_evaluation("llama")

Q01 done
Q02 done
Q03 done
Q04 done
Q05 done
Q06 done
Q07 done
Q08 done
Q09 done
Q10 done
Q11 done
Q12 done
Q13 done
Q14 done
Q15 done
Q16 done
Q17 done
Q18 done
Q19 done
Q20 done
Q21 done
Q22 done
Q23 done
Q24 done
Q25 done
Q26 done
Q27 done
Q28 done
Q29 done
Q30 done
smollm summary: {'context_precision': 0.25, 'context_recall': 0.923, 'mrr': 0.821, 'faithfulness': 0.767, 'relevancy': 0.8, 'correctness': 0.467, 'latency_seconds': 0.475}
Q01 done
Q02 done
Q03 done
Q04 done
Q05 done
Q06 done
Q07 done
Q08 done
Q09 done
Q10 done
Q11 done
Q12 done
Q13 done
Q14 done
Q15 done
Q16 done
Q17 done
Q18 done
Q19 done
Q20 done
Q21 done
Q22 done
Q23 done
Q24 done
Q25 done
Q26 done
Q27 done
Q28 done
Q29 done
Q30 done
llama summary: {'context_precision': 0.25, 'context_recall': 0.923, 'mrr': 0.821, 'faithfulness': 0.833, 'relevancy': 0.8, 'correctness': 0.9, 'latency_seconds': 12.175}


### Compare the two models

In [4]:
import json, pandas as pd
from src.config import RESULTS_DIR

smol_summary = json.loads((RESULTS_DIR / "smollm_summary.json").read_text())
llama_summary = json.loads((RESULTS_DIR / "llama_summary.json").read_text())
pd.DataFrame({"smollm2-135m": smol_summary, "llama3.2-3b": llama_summary})

,smollm2-135m,llama3.2-3b
context_precision,0.250,0.250
context_recall,0.923,0.923
mrr,0.821,0.821
faithfulness,0.767,0.833
relevancy,0.800,0.800
correctness,0.467,0.900
latency_seconds,0.475,12.175


### Failure analysis

Retrieval metrics are the same for both models by design (same retriever, same top-k) - so any gap in
Faithfulness, Relevancy or Correctness is a generation problem, not a retrieval problem.

Real examples seen in this run:
- **Hallucination on out-of-scope questions.** Asked for a tuition fee that isn't in the handbook,
  SmolLM2 invented a number; Llama 3.2-3B correctly replied "Not in handbook."
- **Factual errors on simple lookups.** Asked where the university is located, SmolLM2 answered
  "Karaikal, Karnataka" - fabricated, not in the handbook at all. Llama answered correctly ("Karjat,
  Raigad").
- **Refusal rate.** Across the 4 not-in-handbook questions, Llama refused correctly 4/4 times; SmolLM2
  refused correctly 0/4 times.

At 135M parameters, SmolLM2 has not learned reliable "I don't know" behaviour or careful fact recall,
even when the correct context was retrieved for both models identically.